In [1]:
# ==========================================================================
# AI Image Classification (CNN / TensorFlow)
# Multi-class image classifier — data augmentation + systematic
# hyperparameter tuning — target: 90%+ top-1 accuracy
#
# Dataset: tf_flowers (5 classes: daisy, dandelion, roses, sunflowers, tulips)
#          via tensorflow_datasets — swap for any multi-class dataset you like.
#
# Design note (be ready to explain this in an interview):
# This uses a MobileNetV2 backbone pretrained on ImageNet, with a custom
# classification head that IS trained from scratch, plus the top layers of
# the backbone fine-tuned (unfrozen) on this dataset. This is the standard,
# defensible way to reliably hit 90%+ accuracy on a small dataset (~3,700
# images) without needing days of from-scratch training. If you want a
# 100% from-scratch CNN instead, see the SCRATCH_CNN option near the bottom
# — it will need more epochs/data augmentation and will likely land lower
# than 90% unless trained much longer.
# ==========================================================================

# ---- 0. Setup ----
# NOTE: tensorflow_datasets is deliberately NOT used here — in current Colab
# images it frequently breaks with a protobuf gencode/runtime version clash
# (tensorflow_metadata compiled against a newer protobuf than the active
# runtime). We sidestep it entirely by downloading the dataset directly and
# loading it with image_dataset_from_directory instead.
!pip install -q keras-tuner

import tensorflow as tf
from tensorflow.keras import layers, models
import keras_tuner as kt
import numpy as np
import matplotlib.pyplot as plt
import pathlib

print("TF version:", tf.__version__)
gpu = tf.config.list_physical_devices('GPU')
print("GPU available:", bool(gpu))
# If this prints False: Runtime -> Change runtime type -> GPU, then re-run.

IMG_SIZE = 160
BATCH_SIZE = 32
NUM_CLASSES = 5   # flower_photos has 5 classes

# ---- 1. Load dataset ----
# Official TensorFlow-hosted flowers dataset (daisy, dandelion, roses,
# sunflowers, tulips) — same data as tf_flowers, no tfds dependency.
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file(
    'flower_photos', origin=dataset_url, untar=True
)
data_dir = pathlib.Path(data_dir)

image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"Total images found: {image_count}")

# 70% train / 15% val / 15% test split
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)
val_test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

class_names = train_ds_raw.class_names
print("Classes:", class_names)

# Split the 30% held-out portion in half -> 15% val, 15% test
val_batches = tf.data.experimental.cardinality(val_test_ds_raw)
test_ds_raw = val_test_ds_raw.take(val_batches // 2)
val_ds_raw = val_test_ds_raw.skip(val_batches // 2)

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = train_ds_raw.map(preprocess).cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds_raw.map(preprocess).cache().prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds_raw.map(preprocess).cache().prefetch(tf.data.AUTOTUNE)

# ---- 2. Data augmentation ----
# Flips, rotations, and colour jitter (contrast/brightness) exactly as
# described in the resume bullet.
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),          # rotation jitter
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),           # colour jitter (contrast)
    layers.RandomBrightness(0.2),         # colour jitter (brightness)
], name="data_augmentation")

# Quick visual sanity check of augmentation (optional, run once)
for images, _ in train_ds.take(1):
    plt.figure(figsize=(8, 8))
    for i in range(9):
        augmented = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented[0].numpy())
        plt.axis("off")
    plt.suptitle("Augmentation sample")
    plt.show()

# ---- 3. Model builder (parameterized for hyperparameter tuning) ----
def build_model(hp):
    dense_units = hp.Choice('dense_units', [128, 256, 512])
    dropout_rate = hp.Float('dropout_rate', 0.2, 0.5, step=0.1)
    learning_rate = hp.Choice('learning_rate', [1e-3, 5e-4, 1e-4])
    fine_tune_at = hp.Choice('fine_tune_at', [100, 120, 140])  # layer index to unfreeze from

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = True
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ---- 4. Systematic hyperparameter tuning (Keras Tuner - Hyperband) ----
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=15,
    factor=3,
    directory='cnn_tuning',
    project_name='flowers_cnn'
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop]
)

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters found:")
print(f"  dense_units   = {best_hp.get('dense_units')}")
print(f"  dropout_rate  = {best_hp.get('dropout_rate')}")
print(f"  learning_rate = {best_hp.get('learning_rate')}")
print(f"  fine_tune_at  = {best_hp.get('fine_tune_at')}")

# ---- 5. Train the best model for longer ----
model = tuner.hypermodel.build(best_hp)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[early_stop, reduce_lr]
)

# ---- 6. Evaluate: top-1 accuracy on held-out test set ----
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nFinal Test Top-1 Accuracy: {test_acc * 100:.2f}%")

# ---- 7. Training curves ----
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()
plt.show()

# ---- 8. Confusion matrix (good to show in interview / README) ----
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

# ---- 9. Save model ----
model.save('flower_cnn_classifier.keras')
print("Model saved as flower_cnn_classifier.keras")

# ==========================================================================
# OPTIONAL: SCRATCH_CNN — a fully from-scratch CNN (no pretrained weights)
# Use this only if you specifically need to say "no transfer learning."
# Expect lower accuracy (~75-88% on this dataset) unless trained much longer
# with a larger dataset — worth knowing this tradeoff if asked about it.
# ==========================================================================
SCRATCH_CNN = False
if SCRATCH_CNN:
    def build_scratch_cnn(hp):
        filters_1 = hp.Choice('filters_1', [32, 64])
        filters_2 = hp.Choice('filters_2', [64, 128])
        dropout_rate = hp.Float('dropout_rate', 0.2, 0.5, step=0.1)
        learning_rate = hp.Choice('learning_rate', [1e-3, 5e-4])

        inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = data_augmentation(inputs)
        x = layers.Conv2D(filters_1, 3, activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(filters_2, 3, activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(filters_2 * 2, 3, activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(dropout_rate)(x)
        outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

        model = tf.keras.Model(inputs, outputs)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model
    # Repeat the same tuner.search()/fit()/evaluate() steps above with
    # build_scratch_cnn instead of build_model.

Trial 6 Complete [00h 00m 29s]
val_accuracy: 1.0

Best val_accuracy So Far: 1.0
Total elapsed time: 00h 02m 35s

Search: Running Trial #7

Value             |Best Value So Far |Hyperparameter
256               |512               |dense_units
0.4               |0.3               |dropout_rate
0.0005            |0.0005            |learning_rate
120               |120               |fine_tune_at
2                 |2                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
0                 |0                 |tuner/round

Epoch 1/2
81/81 ━━━━━━━━━━━━━━━━━━━━ 17s 76ms/step - accuracy: 0.9875 - loss: 0.0308 - val_accuracy: 1.0000 - val_loss: 0.0020
Epoch 2/2
56/81 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 1.0000 - loss: 1.6194e-05

KeyboardInterrupt: 

  Using cached deepface-0.0.93-py3-none-any.whl.metadata (30 kB)
  Using cached flask_cors-6.0.1-py3-none-any.whl.metadata (5.3 kB)
  Using cached mtcnn-1.0.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached retina_face-0.0.17-py3-none-any.whl.metadata (10 kB)
  Using cached fire-0.7.0.tar.gz (87 kB)
  Preparing metadata (setup.py) ... done
  Using cached gunicorn-23.0.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached lz4-4.4.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.8 kB)
Using cached deepface-0.0.93-py3-none-any.whl (108 kB)
Using cached flask_cors-6.0.1-py3-none-any.whl (13 kB)
Using cached gunicorn-23.0.0-py3-none-any.whl (85 kB)
Using cached mtcnn-1.0.0-py3-none-any.whl (1.9 MB)
Using cached retina_face-0.0.17-py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.5 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=6ddb39fe93589fe9da838cd0be8edcd9264b0b78d610659b7363c0505

In [ ]:
import gradio as gr
import cv2
from deepface import DeepFace
import numpy as np
from PIL import Image

def analyze_emotions(image):
    img_bgr = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    emotions_summary = []

    try:
        faces = DeepFace.extract_faces(img_path=np.array(image), detector_backend="opencv", enforce_detection=False)

        for i, face in enumerate(faces):
            fa = face["facial_area"]
            x, y = fa.get("x", 0), fa.get("y", 0)
            w, h = fa.get("w", 0), fa.get("h", 0)

            try:
                result = DeepFace.analyze(img_path=np.array(image), actions=["emotion"], enforce_detection=False)[0]
                emotion = result["dominant_emotion"]
                confidence = result["emotion"][emotion]
                emotions_summary.append(f"Face {i+1}: {emotion} ({confidence:.2f}%)")
            except:
                emotion = "Unknown"
                emotions_summary.append(f"Face {i+1}: Unknown")

            cv2.rectangle(img_rgb, (x, y), (x + w, y + h), (0, 200, 100), 2)
            cv2.putText(img_rgb, emotion, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 100), 2)

    except:
        emotions_summary.append("⚠️ No faces detected.")

    final_image = Image.fromarray(img_rgb)
    return final_image, "\n".join(emotions_summary)


custom_css = """
.gradio-container {
  font-family: 'Segoe UI', sans-serif;
  background: #fafafa;
}

.main-page {
  display: flex;
  height: 100vh;
}

.sidebar {
  width: 220px;
  background-color: #ffffff;
  border-right: 1px solid #e0e0e0;
  padding: 30px 20px;
  height: 100vh;
}

.sidebar h2 {
  font-size: 22px;
  font-weight: 700;
  color: #333;
  margin-bottom: 20px;
}

.sidebar ul {
  list-style: none;
  padding: 0;
  font-size: 16px;
}

.sidebar li {
  margin: 12px 0;
  color: #555;
}

.content {
  flex: 1;
  padding: 40px 50px;
  overflow-y: auto;
}

h1.title {
  font-size: 28px;
  font-weight: 700;
  margin-bottom: 10px;
  color: #222;
}

h2.sub {
  font-size: 18px;
  color: #666;
  margin-bottom: 25px;
}

.upload-section, .result-section {
  background: #fff;
  padding: 25px;
  border-radius: 8px;
  box-shadow: 0 0 6px rgba(0,0,0,0.06);
  margin-bottom: 30px;
}

button {
  background-color: #007acc !important;
  color: white !important;
  font-weight: bold !important;
  border-radius: 6px !important;
  padding: 10px 20px !important;
}
"""

# --- Gradio App ---
with gr.Blocks(css=custom_css, title="Facial Emotion Recognition") as app:
    gr.HTML("""
    <div class='main-page'>
      <div class='sidebar'>
        <h2>Face Emotion</h2>
        <ul>
          <li>Overview</li>
          <li>Run</li>
          <li>Output</li>
          <li>About</li>
        </ul>
      </div>
      <div class='content'>
        <h1 class='title'>Facial Emotion Recognition</h1>
        <h2 class='sub'>AI-powered system to detect human emotions like Happy, Sad, Angry from images using Deep Learning.</h2>

        <div style="margin-top: 30px; font-size: 15px; color: #444; line-height: 1.7;">


        <h3>📌 Overview</h3>
        <p>
        Facial Emotion Recognition is a computer vision project that uses deep learning to identify the <b>emotional state</b> of people in images.
        By applying pre-trained models like <code>DeepFace</code>, we can analyze facial expressions and categorize them into emotions like <b>Happy</b>, <b>Sad</b>, <b>Angry</b>, <b>Surprise</b>, and more.
        </p>
        <p>
        This system has real-world applications in <b>mental health monitoring</b>, <b>education</b>, <b>marketing analytics</b>, and <b>human-computer interaction</b>.
        </p>


        <h3>🚀 Run Instructions</h3>
        <ul>
          <li>Upload a facial image (single or group)</li>
          <li>Click <b>"Analyze Emotions"</b></li>
          <li>The app detects each face and highlights its dominant emotion</li>
        </ul>
        <p>
        Green boxes indicate detected faces. Labels are drawn with the identified emotion (e.g., <code>Happy</code>, <code>Sad</code>).
        </p>


        <h3>🖼️ Output</h3>
        <p>
        After processing the image, the app shows:
        <ul>
          <li>✅ Annotated image with bounding boxes and emotion labels</li>
          <li>🧠 A summary of all detected emotions with confidence scores</li>
        </ul>
        </p>
        <p>Example Output:
        <pre>
Face 1: Happy (92.14%)
Face 2: Sad (66.87%)
        </pre>
        </p>

        <!-- ABOUT -->
        <h3>👨‍💻 About the Project</h3>
        <p>
        This project was built using:
        <ul>
          <li><b>DeepFace</b> - for face and emotion recognition</li>
          <li><b>OpenCV</b> - for image annotation and processing</li>
          <li><b>Gradio</b> - for creating a modern, web-based UI</li>
        </ul>

        </p>
        <p>
        This tool is ideal for students, researchers, and developers interested in AI-powered emotion classification. Future versions could integrate real-time webcam support, emotion graphs, or even log analysis.
        </p>

        <p style="margin-top: 25px; font-style: italic; font-size: 14px;">
        💡 Developed as a smart AI interface demo for interviews, workshops, and portfolio presentations.
        </p>

        </div>
    """)


    with gr.Column(elem_classes="upload-section"):
        image_input = gr.Image(type="pil", label="📤 Upload an Image")
        analyze_btn = gr.Button("🔍 Analyze Emotions")


    with gr.Column(elem_classes="result-section"):
        image_output = gr.Image(type="pil", label="📸 Annotated Output")
        emotion_output = gr.Textbox(label="📝 Detected Emotions", lines=8)


    analyze_btn.click(analyze_emotions, inputs=image_input, outputs=[image_output, emotion_output])

    gr.HTML("</div></div>")

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f204a7fe62abcd7d3e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install gradio
!pip install deepface
!pip install ultralytics
!pip install gradio ultralytics deepface opencv-python matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.1 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=fb3adee0ad9352612d60c3900b47893b257466fe78cb7115e140a3a363b42d85
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6